# End-to-end ML project: select and train a model

**Author:** [firstname] [lastname]

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

Select and train a model, and a first look at fine-tuning. Code
along with the slides: each section below matches one code-along in the
slide deck.

Run the setup cell first. It loads the housing data and rebuilds
`housing`, `housing_labels` and the final transformer `preprocessing`
from part 7.

Code-alongs 1 and 2 are filled in, as typed in class. Run the setup
cell and both code-alongs, check the expected output, then continue
with code-along 3.

In [ ]:
# Setup: rebuild the training set and the final transformer from parts 3 to 7
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

# training and test set (part 3)
housing_full = pd.read_csv("https://tinyurl.com/housing-csv")
housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5])
strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2,
    stratify=housing_full["income_cat"],
    random_state=42)
for df in (strat_train_set, strat_test_set):
    df.drop("income_cat", axis=1, inplace=True)
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

# custom transformer (part 7)
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

# building blocks of the final transformer (part 7)
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"))

# the final transformer (part 7)
preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                           "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)

print(housing.shape, preprocessing.fit_transform(housing).shape)

Expected output:

```
(16512, 9) (16512, 24)
```

## Code-along 1. Train a linear regression

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = make_pipeline(preprocessing, LinearRegression())

lin_reg.fit(housing, housing_labels) # X_train and y_train

housing_pred = lin_reg.predict(housing)  # y_pred

print(housing_pred[:5].round(-2))
print(housing_labels.iloc[:5].values)

Expected output:

```
[246000. 372700. 135700.  91400. 330900.]
[458300. 483800. 101700.  96100. 361800.]
```

## Code-along 2. RMSE on the training set

In [ ]:
from sklearn.metrics import root_mean_squared_error
lin_rmse = root_mean_squared_error(housing_labels, housing_pred)
print(round(lin_rmse))

Expected output:

```
68973
```

## Code-along 3. A decision tree

## Code-along 4. Cross-validate the tree

## Code-along 5. Cross-validate the linear regression

## Code-along 6. A random forest

## Practice: cross-validate a k-nearest neighbors regressor

Put a `KNeighborsRegressor` (default hyperparameters) behind
`preprocessing`, run 10-fold cross-validation, and compare its mean RMSE
with the three models. Where does it land?